In [1]:
import re
from collections import Counter

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
def preprocess_text(text: str) -> list[str]:
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s.,!?;:'\"()-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = re.findall(r"\w+|[.,!?;:'\"()-]", text)
    return tokens

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

In [3]:
class Vocab:
    def __init__(self, min_freq: int = 2, max_size: int | None = 20000):
        self.min_freq = min_freq
        self.max_size = max_size

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"

        self.stoi = {self.pad_token: 0, self.unk_token: 1}
        self.itos = [self.pad_token, self.unk_token]

    def build(self, tokenized_texts: list[list[str]]) -> None:
        counter = Counter()
        for tokens in tokenized_texts:
            counter.update(tokens)

        items = [(tok, freq) for tok, freq in counter.items() if freq >= self.min_freq]
        items.sort(key=lambda x: (-x[1], x[0]))

        if self.max_size is not None:
            max_new_tokens = self.max_size - len(self.itos)
            items = items[:max_new_tokens]

        for token, _ in items:
            self.stoi[token] = len(self.itos)
            self.itos.append(token)

    def encode(self, tokens: list[str]) -> list[int]:
        unk_idx = self.stoi[self.unk_token]
        return [self.stoi.get(tok, unk_idx) for tok in tokens]

    def decode(self, ids: list[int]) -> list[str]:
        return [self.itos[i] if 0 <= i < len(self.itos) else self.unk_token for i in ids]

    @property
    def pad_idx(self) -> int:
        return self.stoi[self.pad_token]

    @property
    def unk_idx(self) -> int:
        return self.stoi[self.unk_token]

    def __len__(self) -> int:
        return len(self.itos)


In [4]:
class IMDBDataset(Dataset):
    def __init__(
        self,
        texts: list[str],
        labels: list[int],
        vocab: Vocab,
        max_len: int = 50
    ):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        label = self.labels[idx]

        tokens = preprocess_text(text)
        input_ids = self.vocab.encode(tokens)
        input_ids = input_ids[:self.max_len]
        attention_mask = [1] * len(input_ids)
        pad_length = self.max_len - len(input_ids)
        if pad_length > 0:
            input_ids = input_ids + [self.vocab.pad_idx] * pad_length
            attention_mask = attention_mask + [0] * pad_length

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long),
        }

In [5]:
def load_imdb_dataframe(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df[["review", "sentiment"]].dropna().copy()
    label_map = {"negative": 0, "positive": 1}
    df["label"] = df["sentiment"].map(label_map)
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)
    return df


def build_splits(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42
):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=random_state
    )

    relative_val_size = val_size / (1 - test_size)

    train_df, val_df = train_test_split(
        train_df,
        test_size=relative_val_size,
        stratify=train_df["label"],
        random_state=random_state
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [6]:
def create_imdb_dataloaders(
    csv_path: str,
    batch_size: int = 32,
    max_len: int = 50,
    min_freq: int = 2,
    max_vocab_size: int = 20000,
    random_state: int = 42
):
    df = load_imdb_dataframe(csv_path)
    train_df, val_df, test_df = build_splits(df, random_state=random_state)
    train_tokens = [preprocess_text(text) for text in train_df["review"].tolist()]
    vocab = Vocab(min_freq=min_freq, max_size=max_vocab_size)
    vocab.build(train_tokens)

    train_dataset = IMDBDataset(
        texts=train_df["review"].tolist(),
        labels=train_df["label"].tolist(),
        vocab=vocab,
        max_len=max_len
    )
    val_dataset = IMDBDataset(
        texts=val_df["review"].tolist(),
        labels=val_df["label"].tolist(),
        vocab=vocab,
        max_len=max_len
    )
    test_dataset = IMDBDataset(
        texts=test_df["review"].tolist(),
        labels=test_df["label"].tolist(),
        vocab=vocab,
        max_len=max_len
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, vocab

In [7]:
if __name__ == "__main__":
    csv_path = "data/IMDB.csv"

    train_loader, val_loader, test_loader, vocab = create_imdb_dataloaders(
        csv_path=csv_path,
        batch_size=32,
        max_len=50,
        min_freq=2,
        max_vocab_size=20000,
        random_state=42
    )

    print(f"Vocab size: {len(vocab)}")
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")

    batch = next(iter(train_loader))
    print("input_ids shape:", batch["input_ids"].shape)
    print("attention_mask shape:", batch["attention_mask"].shape)
    print("label shape:", batch["label"].shape)

    sample_ids = batch["input_ids"][0].tolist()
    sample_tokens = vocab.decode(sample_ids[:30])
    print("First 30 decoded tokens:", sample_tokens)
    print("Label:", batch["label"][0].item())

Vocab size: 20000
Train batches: 1094
Val batches: 157
Test batches: 313
input_ids shape: torch.Size([32, 200])
attention_mask shape: torch.Size([32, 200])
label shape: torch.Size([32])
First 30 decoded tokens: ['watchers', 'is', 'a', 'fun', 'movie', 'if', 'it', "'", 's', 'not', 'taken', 'too', 'seriously', ',', 'the', 'novel', 'written', 'by', 'dean', 'r', '.', 'koontz', 'is', 'obviously', 'a', 'lot', 'better', 'but', 'the', 'movie']
Label: 1


## LSTM

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMBaseline(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
        embedding_dropout: float = 0.2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(embedding_dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)
        
        output, (hidden, cell) = self.lstm(embedded)

        final_hidden = hidden[-1]
        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)
        return logits


def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


def run_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 15,
    embed_dim: int = 50,
    hidden_dim: int = 100,
    lr: float = 1e-4,
    dropout: float = 0.3,
    embedding_dropout: float = 0.2,
    weight_decay: float = 1e-4,
    patience: int = 3,
):
    device = get_device()
    print("Using device:", device)

    model = LSTMBaseline(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=dropout,
        embedding_dropout=embedding_dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_acc = 0.0
    best_model_state = None
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

## BILSTM

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

class BiLSTMBaseline(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
        embedding_dropout: float = 0.2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(embedding_dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)
        
        output, (hidden, cell) = self.lstm(embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)
        return logits

def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

def run_bilstm_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 15,
    embed_dim: int = 50,
    hidden_dim: int = 100,
    lr: float = 1e-4,
    dropout: float = 0.3,
    embedding_dropout: float = 0.2,
    weight_decay: float = 1e-4,
    patience: int = 3,
):
    device = get_device()
    print("Using device:", device)

    model = BiLSTMBaseline(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=dropout,
        embedding_dropout=embedding_dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_acc = 0.0
    best_model_state = None
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

## LSTM + Attention

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs, attention_mask=None):
        scores = self.attn(lstm_outputs)
        scores = scores.squeeze(-1)

        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_outputs).squeeze(1)

        return context, attn_weights

class LSTMAttentionModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
        embedding_dropout: float = 0.2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(embedding_dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        self.attention = AttentionLayer(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask=None, return_attention=False):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)
        
        outputs, (hidden, cell) = self.lstm(embedded)

        context, attn_weights = self.attention(outputs, attention_mask)
        context = self.dropout(context)

        logits = self.fc(context)

        if return_attention:
            return logits, attn_weights
        return logits

In [19]:
def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch_attention(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate_attention(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

def run_lstm_attention_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 15,
    embed_dim: int = 50,
    hidden_dim: int = 100,
    lr: float = 1e-4,
    dropout: float = 0.3,
    embedding_dropout: float = 0.2,
    weight_decay: float = 1e-4,
    patience: int = 3,
):
    device = get_device()
    print("Using device:", device)

    model = LSTMAttentionModel(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=dropout,
        embedding_dropout=embedding_dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_acc = 0.0
    best_model_state = None
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch_attention(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate_attention(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate_attention(model, test_loader, criterion, device)

    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

## BiLSTM + Attention

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim


class BiAttentionLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_outputs, attention_mask=None):
        scores = self.attn(lstm_outputs)
        scores = scores.squeeze(-1)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_outputs).squeeze(1)

        return context, attn_weights


class BiLSTMAttentionModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
        embedding_dropout: float = 0.2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(embedding_dropout)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.attention = BiAttentionLayer(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None, return_attention=False):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)
        
        outputs, (hidden, cell) = self.lstm(embedded)

        context, attn_weights = self.attention(outputs, attention_mask)
        context = self.dropout(context)

        logits = self.fc(context)

        if return_attention:
            return logits, attn_weights
        return logits

In [22]:
def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch_attention(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate_attention(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

def run_bilstm_attention_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 15,
    embed_dim: int = 50,
    hidden_dim: int = 100,
    lr: float = 1e-4,
    dropout: float = 0.3,
    embedding_dropout: float = 0.2,
    weight_decay: float = 1e-4,
    patience: int = 3,
):
    device = get_device()
    print("Using device:", device)

    model = BiLSTMAttentionModel(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=dropout,
        embedding_dropout=embedding_dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_acc = 0.0
    best_model_state = None
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch_attention(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate_attention(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate_attention(model, test_loader, criterion, device)

    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

## Average Pooling

In [ ]:
import torch
import torch.nn as nn


class MaskedAveragePooling(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, lstm_outputs, attention_mask=None):
        if attention_mask is None:
            return lstm_outputs.mean(dim=1)
        mask = attention_mask.unsqueeze(-1).float()
        masked_outputs = lstm_outputs * mask
        summed = masked_outputs.sum(dim=1)
        lengths = mask.sum(dim=1)
        lengths = torch.clamp(lengths, min=1e-8)
        pooled = summed / lengths
        return pooled
    
class BiLSTMAvgPoolModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
        embedding_dropout: float = 0.2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(embedding_dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.pool = MaskedAveragePooling()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)
        
        outputs, (hidden, cell) = self.lstm(embedded)

        pooled = self.pool(outputs, attention_mask)
        pooled = self.dropout(pooled)

        logits = self.fc(pooled)
        return logits

In [ ]:
def train_one_epoch_attention(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def evaluate_attention(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

import torch.optim as optim

def run_bilstm_avgpool_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 15,
    embed_dim: int = 50,
    hidden_dim: int = 100,
    lr: float = 1e-4,
    dropout: float = 0.3,
    embedding_dropout: float = 0.2,
    weight_decay: float = 1e-4,
    patience: int = 3,
):
    device = get_device()
    print("Using device:", device)

    model = BiLSTMAvgPoolModel(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=dropout,
        embedding_dropout=embedding_dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_acc = 0.0
    best_model_state = None
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch_attention(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate_attention(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate_attention(model, test_loader, criterion, device)

    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

### Data

In [14]:
if __name__ == "__main__":
    train_loader, val_loader, test_loader, vocab = create_imdb_dataloaders(
        csv_path="data/IMDB.csv",
        batch_size=32,
        max_len=50,
        min_freq=2,
        max_vocab_size=20000,
        random_state=42
    )

### LSTM result

In [15]:
lstm_model = run_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=15,
    embed_dim=50,
    hidden_dim=100,
    lr=1e-4,
    dropout=0.3,
    embedding_dropout=0.2,
    weight_decay=1e-4,
    patience=3,
)

Using device: cpu
Epoch 01 | Train Loss: 0.6908 | Train Acc: 0.5305 | Val Loss: 0.6821 | Val Acc: 0.5574
Epoch 02 | Train Loss: 0.6666 | Train Acc: 0.5954 | Val Loss: 0.6471 | Val Acc: 0.6456
Epoch 03 | Train Loss: 0.5954 | Train Acc: 0.6995 | Val Loss: 0.6773 | Val Acc: 0.5634
Epoch 04 | Train Loss: 0.5360 | Train Acc: 0.7215 | Val Loss: 0.4608 | Val Acc: 0.7892
Epoch 05 | Train Loss: 0.3538 | Train Acc: 0.8525 | Val Loss: 0.3508 | Val Acc: 0.8520

Best Val Acc: 0.8520
Test Loss: 0.3552 | Test Acc: 0.8523


### BILSTM result

In [16]:
bilstm_model = run_bilstm_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=15,
    embed_dim=50,
    hidden_dim=100,
    lr=1e-4,
    dropout=0.3,
    embedding_dropout=0.2,
    weight_decay=1e-4,
    patience=3,
)

Using device: cpu
Epoch 01 | Train Loss: 0.6482 | Train Acc: 0.6160 | Val Loss: 0.5498 | Val Acc: 0.7282
Epoch 02 | Train Loss: 0.3897 | Train Acc: 0.8291 | Val Loss: 0.3343 | Val Acc: 0.8526
Epoch 03 | Train Loss: 0.2613 | Train Acc: 0.8957 | Val Loss: 0.3338 | Val Acc: 0.8622
Epoch 04 | Train Loss: 0.1797 | Train Acc: 0.9322 | Val Loss: 0.3585 | Val Acc: 0.8652
Epoch 05 | Train Loss: 0.1094 | Train Acc: 0.9618 | Val Loss: 0.3973 | Val Acc: 0.8616

Best Val Acc: 0.8652
Test Loss: 0.3697 | Test Acc: 0.8600


### LSTM + Attention result

In [20]:
lstm_attn_model = run_lstm_attention_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=15,
    embed_dim=50,
    hidden_dim=100,
    lr=1e-4,
    dropout=0.3,
    embedding_dropout=0.2,
    weight_decay=1e-4,
    patience=3,
)

Using device: cpu
Epoch 01 | Train Loss: 0.4257 | Train Acc: 0.7970 | Val Loss: 0.2934 | Val Acc: 0.8684
Epoch 02 | Train Loss: 0.2591 | Train Acc: 0.8920 | Val Loss: 0.2679 | Val Acc: 0.8824
Epoch 03 | Train Loss: 0.1853 | Train Acc: 0.9297 | Val Loss: 0.2730 | Val Acc: 0.8862
Epoch 04 | Train Loss: 0.1152 | Train Acc: 0.9582 | Val Loss: 0.3564 | Val Acc: 0.8838
Epoch 05 | Train Loss: 0.0564 | Train Acc: 0.9814 | Val Loss: 0.4496 | Val Acc: 0.8764

Best Val Acc: 0.8862
Test Loss: 0.2909 | Test Acc: 0.8834


### BiLSTM + Attention result

In [23]:
bilstm_attn_model = run_bilstm_attention_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=15,
    embed_dim=50,
    hidden_dim=100,
    lr=1e-4,
    dropout=0.3,
    embedding_dropout=0.2,
    weight_decay=1e-4,
    patience=3,
)

Using device: cpu
Epoch 01 | Train Loss: 0.4089 | Train Acc: 0.8058 | Val Loss: 0.3075 | Val Acc: 0.8600
Epoch 02 | Train Loss: 0.2513 | Train Acc: 0.8973 | Val Loss: 0.2679 | Val Acc: 0.8882
Epoch 03 | Train Loss: 0.1741 | Train Acc: 0.9330 | Val Loss: 0.2865 | Val Acc: 0.8768
Epoch 04 | Train Loss: 0.1032 | Train Acc: 0.9642 | Val Loss: 0.3343 | Val Acc: 0.8852
Epoch 05 | Train Loss: 0.0465 | Train Acc: 0.9857 | Val Loss: 0.4760 | Val Acc: 0.8764

Best Val Acc: 0.8882
Test Loss: 0.2810 | Test Acc: 0.8794


### BiLSTM + Average Pooling result

In [ ]:
bilstm_avgpool_model = run_bilstm_avgpool_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=15,
    embed_dim=50,
    hidden_dim=100,
    lr=1e-4,
    dropout=0.3,
    embedding_dropout=0.2,
    weight_decay=1e-4,
    patience=3,
)